# 01_17 — Caso haja problemas: enviar diagnóstico

> **Material de familiarização com o JupyterLab do SimServ.**  
> Estes notebooks foram pensados para ficar em uma pasta de somente leitura, por exemplo `/sobre_ambientes_simserv`.  
> Para editar, testar livremente ou salvar resultados, faça uma cópia para sua área de trabalho, como `/home/jovyan/ambiente_laboratorio`.

## Objetivo

Gerar um arquivo de diagnóstico simples para enviar ao professor, monitor ou administrador.

O arquivo gerado evita mensagens vagas como “não funcionou” e ajuda a identificar problemas de permissão, kernel, bibliotecas ou IA local.

In [1]:
from pathlib import Path
import sys
import platform
import datetime
import os
import getpass
import json
from urllib.request import urlopen

In [2]:
def encontrar_area_de_trabalho():
    candidatas = [
        Path.home() / "laboratorio",
        Path.home() / "ambiente_laboratorio",
        Path("/home/jovyan/laboratorio"),
        Path("/home/jovyan/ambiente_laboratorio"),
        Path.cwd(),
    ]
    for pasta in candidatas:
        try:
            if pasta.exists() and os.access(pasta, os.W_OK):
                return pasta
        except Exception:
            pass
    raise RuntimeError("Não encontrei uma pasta gravável para salvar o diagnóstico.")

AREA_TRABALHO = encontrar_area_de_trabalho()
print("O diagnóstico será salvo em:", AREA_TRABALHO)

O diagnóstico será salvo em: /home/jovyan/ambiente_laboratorio


In [10]:
def testar_ollama():
    for base in ["http://ollama:11434", "http://127.0.0.1:11434", "http://localhost:11434"]:
        try:
            with urlopen(base + "/api/tags", timeout=5) as resposta:
                dados = json.loads(resposta.read().decode("utf-8"))
            modelos = [m.get("name", "sem_nome") for m in dados.get("models", [])]
            return True, base, modelos
        except Exception:
            pass
    return False, "", []

def testar_bibliotecas(nomes):
    saida = []
    for nome in nomes:
        try:
            modulo = __import__(nome)
            versao = getattr(modulo, "__version__", "sem versão")
            saida.append((nome, True, versao))
        except Exception as erro:
            saida.append((nome, False, str(erro)))
    return saida

In [12]:
agora = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
arquivo = AREA_TRABALHO / f"diagnostico_jupyterlab_simserv_{agora}.txt"

linhas = []
linhas.append("Diagnóstico JupyterLab SimServ")
linhas.append("=" * 34)
linhas.append(f"Data/hora: {datetime.datetime.now()}")
linhas.append(f"Usuário do sistema: {getpass.getuser()}")
linhas.append(f"Python: {sys.version}")
linhas.append(f"Executável Python: {sys.executable}")
linhas.append(f"Sistema: {platform.platform()}")
linhas.append(f"Pasta atual: {Path.cwd()}")
linhas.append(f"HOME: {Path.home()}")
linhas.append("")

linhas.append("Pastas e permissões")
linhas.append("-" * 22)
for pasta in [Path.cwd(), Path.home(), Path.home() / "ambiente_laboratorio", Path.home() / "laboratorio", Path.home() / "shared", Path.home() / "exercicios_e_recursos", Path("/sobre_ambientes_simserv")]:
    existe = pasta.exists()
    leitura = os.access(pasta, os.R_OK) if existe else False
    escrita = os.access(pasta, os.W_OK) if existe else False
    linhas.append(f"{pasta}: existe={existe}; leitura={leitura}; escrita={escrita}")
    linhas.append("")
    
    linhas.append("Bibliotecas principais")
    linhas.append("-" * 24)
for nome, ok, detalhe in testar_bibliotecas(["numpy", "pandas", "matplotlib", "requests", "IPython", "ipykernel"]):
    linhas.append(f"{nome}: {'OK' if ok else 'ERRO'}; {detalhe}")
    linhas.append("")
    
    ok_ia, base_ia, modelos = testar_ollama()
    linhas.append("IA local Ollama")
    linhas.append("-" * 15)
    linhas.append(f"Disponível: {ok_ia}")
    if ok_ia:
        linhas.append(f"URL que respondeu: {base_ia}")
        linhas.append("Modelos: " + (", ".join(modelos) if modelos else "nenhum listado"))
        linhas.append("")
    
        linhas.append("Observação do aluno")
        linhas.append("-" * 19)
        linhas.append("Descreva aqui o que você tentou fazer e o que aconteceu.")
        
        #arquivo.write_text(".join(linhas), encoding="utf-8")
        arquivo.write_text("\n".join(linhas), encoding="utf-8")
        
        print("Arquivo de diagnóstico criado com sucesso:")
        print(arquivo)
        print("Envie esse arquivo ao professor, monitor ou administrador, junto com uma descrição breve do problema.")

Arquivo de diagnóstico criado com sucesso:
/home/jovyan/ambiente_laboratorio/diagnostico_jupyterlab_simserv_2026-05-20_01-57-13.txt
Envie esse arquivo ao professor, monitor ou administrador, junto com uma descrição breve do problema.
Arquivo de diagnóstico criado com sucesso:
/home/jovyan/ambiente_laboratorio/diagnostico_jupyterlab_simserv_2026-05-20_01-57-13.txt
Envie esse arquivo ao professor, monitor ou administrador, junto com uma descrição breve do problema.
Arquivo de diagnóstico criado com sucesso:
/home/jovyan/ambiente_laboratorio/diagnostico_jupyterlab_simserv_2026-05-20_01-57-13.txt
Envie esse arquivo ao professor, monitor ou administrador, junto com uma descrição breve do problema.
Arquivo de diagnóstico criado com sucesso:
/home/jovyan/ambiente_laboratorio/diagnostico_jupyterlab_simserv_2026-05-20_01-57-13.txt
Envie esse arquivo ao professor, monitor ou administrador, junto com uma descrição breve do problema.
Arquivo de diagnóstico criado com sucesso:
/home/jovyan/ambiente